In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg
import pandas as pd
import matplotlib.pyplot as plt

# 1. Start Spark
spark = SparkSession.builder.master("local[*]").appName("RetailMart_Q5").config("spark.driver.memory", "4g").getOrCreate()
from google.colab import drive
drive.mount('/content/drive')

# 2. Load Data
print(" Loading data...")
df = spark.read.parquet("/content/drive/MyDrive/cleaned_retailmart_master.parquet")
df = df.dropna(subset=["churn"]).withColumn("churn_num", when(col("churn") == "Yes", 1.0).otherwise(0.0))
train, test = df.randomSplit([0.7, 0.3], seed=42)

# 3. Build Pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

numeric_cols = ["customer_age", "loyalty_years", "num_visits", "unit_price", "quantity", "total_amount"]
categorical_cols = ["customer_gender", "product_category", "payment_method", "store_location"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in categorical_cols]
assembler = VectorAssembler(inputCols=numeric_cols + [c+"_ohe" for c in categorical_cols], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
prep_model = prep_pipeline.fit(train)

train_t = prep_model.transform(train).cache()
test_t = prep_model.transform(test).cache()
_ = train_t.count() # Force cache
print(" Data prepped and cached!")

Mounted at /content/drive
 Loading data...
 Data prepped and cached!


In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier, NaiveBayes, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="churn_num", metricName="f1")

# 1. Decision Tree
dt = DecisionTreeClassifier(labelCol="churn_num", featuresCol="features", maxDepth=5)
dt_f1 = evaluator.evaluate(dt.fit(train_t).transform(test_t))

# 2. SVM
svm = LinearSVC(labelCol="churn_num", featuresCol="features", maxIter=100, regParam=0.1)
svm_f1 = evaluator.evaluate(svm.fit(train_t).transform(test_t))

# 3. Naive Bayes (using raw_features because multinomial requires non-negative values)
nb = NaiveBayes(labelCol="churn_num", featuresCol="raw_features", modelType="multinomial")
nb_f1 = evaluator.evaluate(nb.fit(train_t).transform(test_t))

print(f" Decision Tree F1: {dt_f1:.4f}")
print(f" SVM F1:           {svm_f1:.4f}")
print(f" Naive Bayes F1:   {nb_f1:.4f}")

 Decision Tree F1: 0.7269
 SVM F1:           0.6791
 Naive Bayes F1:   0.6658


In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Cluster on BEHAVIORAL features only (no demographics, no target variable)
seg_features = ["customer_age", "loyalty_years", "num_visits", "total_amount"]
seg_assembler = VectorAssembler(inputCols=seg_features, outputCol="seg_features")
train_seg = seg_assembler.transform(train).cache()
_ = train_seg.count()

print(" Testing K values for Elbow Method...")
for k in [3, 4, 5, 6]:
    km = KMeans(featuresCol="seg_features", k=k, seed=42).fit(train_seg)
    print(f"  k={k}: WSSSE = {km.summary.trainingCost:,.0f}")

# Train with K=4
K_CHOICE = 4
kmeans_model = KMeans(featuresCol="seg_features", k=K_CHOICE, seed=42).fit(train_seg)
segmented = kmeans_model.transform(train_seg)

ce = ClusteringEvaluator(featuresCol="seg_features", metricName="silhouette")
silhouette = ce.evaluate(segmented)
print(f"\n✅ K-Means Silhouette Score: {silhouette:.4f}")

 Testing K values for Elbow Method...
  k=3: WSSSE = 10,883,347,005
  k=4: WSSSE = 6,687,677,441
  k=5: WSSSE = 4,504,533,359
  k=6: WSSSE = 3,424,003,641

✅ K-Means Silhouette Score: 0.8348


In [ ]:
segments = []
total_train = train_seg.count()

# Calculate stats for each cluster
for i in range(K_CHOICE):
    seg = segmented.filter(col("prediction") == i)
    n = seg.count()
    stats = seg.selectExpr(
        "avg(customer_age) AS age",
        "avg(loyalty_years) AS loyalty",
        "avg(num_visits) AS visits",
        "avg(total_amount) AS spend"
    ).first()
    churn = seg.filter(col("churn_num") == 1.0).count() / n * 100

    segments.append({
        "segment": i, "n": n, "age": stats['age'], "loyalty": stats['loyalty'],
        "visits": stats['visits'], "spend": stats['spend'], "churn": churn
    })

# Sort by highest churn rate first
segments_sorted = sorted(segments, key=lambda x: x['churn'], reverse=True)
personas = ["AT-RISK", "EARLY-STAGE", "MID-STABLE", "LOYAL PREMIUM"]

print(f"{'Persona':<15} | {'Customers':>9} | {'Churn%':>7} | {'Avg Spend':>10} | {'Loyalty':>8}")
print("-" * 65)

for idx, s in enumerate(segments_sorted):
    s["persona"] = personas[idx] if idx < len(personas) else f"SEG-{idx}"
    print(f"{s['persona']:<15} | {s['n']:>9,} | {s['churn']:>6.1f}% | ${s['spend']:>9.2f} | {s['loyalty']:>4.1f} yrs")

Persona         | Customers |  Churn% |  Avg Spend |  Loyalty
-----------------------------------------------------------------
AT-RISK         |   500,902 |   22.9% | $    70.25 |  2.4 yrs
EARLY-STAGE     |   111,502 |   20.4% | $   352.08 |  2.4 yrs
MID-STABLE      |    26,932 |   20.0% | $   958.14 |  2.4 yrs
LOYAL PREMIUM   |     5,360 |   19.9% | $  2162.19 |  2.5 yrs


In [ ]:
top_seg = segments_sorted[0]

campaigns = {
    "AT-RISK":       "First-100-Days onboarding + immediate retention discount",
    "EARLY-STAGE":   "Mid-term check-in + personalised value communication",
    "MID-STABLE":    "Annual contract upsell + loyalty rewards",
    "LOYAL PREMIUM": "VIP perks + premium cross-sell (no discounting)"
}

print(f"""
============================================================
Q5 — WHICH SEGMENTS TO TARGET? — FINAL ANSWER
============================================================
Method: K-Means (k=4, Silhouette={silhouette:.4f})
Best Supervised Model: Decision Tree (F1={dt_f1:.4f})

TOP-PRIORITY SEGMENT: {top_seg['persona']}
  • Size: {top_seg['n']:,} customers
  • Churn Risk: {top_seg['churn']:.1f}%
  • Recommended Campaign: {campaigns[top_seg['persona']]}

ALL SEGMENTS & CAMPAIGNS:""")

for s in segments_sorted:
    print(f"  {s['persona']:<15} ({s['n']:,} cust) -> {campaigns[s['persona']]}")
print("============================================================")


Q5 — WHICH SEGMENTS TO TARGET? — FINAL ANSWER
Method: K-Means (k=4, Silhouette=0.8348)
Best Supervised Model: Decision Tree (F1=0.7269)

TOP-PRIORITY SEGMENT: AT-RISK
  • Size: 500,902 customers
  • Churn Risk: 22.9%
  • Recommended Campaign: First-100-Days onboarding + immediate retention discount

ALL SEGMENTS & CAMPAIGNS:
  AT-RISK         (500,902 cust) -> First-100-Days onboarding + immediate retention discount
  EARLY-STAGE     (111,502 cust) -> Mid-term check-in + personalised value communication
  MID-STABLE      (26,932 cust) -> Annual contract upsell + loyalty rewards
  LOYAL PREMIUM   (5,360 cust) -> VIP perks + premium cross-sell (no discounting)
